# Retrieval Evaluation

This notebook compares BM25, FAISS, and hybrid retrieval using a small set of representative product queries.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.retrieval import HybridRetriever

pd.set_option("display.max_colwidth", 100)

retriever = HybridRetriever()

Loading product data...
Loading BM25 index...
Loading FAISS index...
Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Hybrid retriever ready.


## 1. Evaluation Queries

Six queries are used to cover different product search requirements.

In [2]:
queries = [
    "iPhone",
    "wireless earbuds with good battery life",
    "comfortable earbuds for running with good battery life",
    "waterproof wireless earbuds for sports",
    "iPhone case with good grip",
    "earbuds with good battery life but poor Bluetooth connectivity is unacceptable"
]

queries

['iPhone',
 'wireless earbuds with good battery life',
 'comfortable earbuds for running with good battery life',
 'waterproof wireless earbuds for sports',
 'iPhone case with good grip',
 'earbuds with good battery life but poor Bluetooth connectivity is unacceptable']

## 2. BM25 Results

BM25 retrieves products based on keyword matching.

In [3]:
bm25_results = {}

for query in queries:
    bm25_results[query] = retriever.bm25_search(query, top_k=5)

for query in queries:
    print(f"\nQUERY: {query}")
    for rank, result in enumerate(bm25_results[query], start=1):
        product = retriever.product_lookup[result["parent_asin"]]
        print(
            f"{rank}. {product['title']} "
            f"({result['parent_asin']})"
        )


QUERY: iPhone
1. Apple Compatible iPhone 5c Case iPhone 5c Protective Case Clear iPhone 5c Case iPhone 5c Cover Case iPhone 5c Green Case by iPhone 5c Silicone Case iPhone 5c Wallet Thin Case by Cable And Case (B00FJIZK8A)
2. iPhone 6 Plus Case, iPhone 6 Plus or 6S Plus Armor Cases 6 Plus Tough Rugged Shockproof Armorbox Dual Layer Hybrid Hard or Soft Slim Protective Case by Cable and Case by White Armor Case (B00NOGWXFC)
3. iPhone 6S Case, iPhone 6 Wallet Case, Firefish [Card Slots] [Kickstand] Flip Folio Wallet Case Synthetic Leather Shell Scratch Resistant Protective Cover for Apple iPhone 6/6S 4.7"-Butterfly (B01LW7QCKO)
4. iPhone 6S Plus Case, iPhone 6 Plus Case by Cable And Case - Raised Screen Protector - Compatible Apple iPhone 6S Plus Case with Kick Stand - Phone Cases for iPhone 6s Plus- (RED) (B00NOGWZKK)
5. Small Portable Charger 5200mAh for iPhone, Ultra Compact 20W PD Fast Charging Power Bank,Cute Mini Battery Pack Backup Charger Compatible with iPhone 14/14 Pro Max/13/1

## 3. FAISS Results

FAISS retrieves products using semantic similarity between the query and product text.

In [4]:
faiss_results = {}

for query in queries:
    faiss_results[query] = retriever.semantic_search(query, top_k=5)

for query in queries:
    print(f"\nQUERY: {query}")
    for rank, result in enumerate(faiss_results[query], start=1):
        product = retriever.product_lookup[result["parent_asin"]]
        print(
            f"{rank}. {product['title']} "
            f"({result['parent_asin']})"
        )


QUERY: iPhone
1. Apple iPhone 6 64 GB AT&T, Silver (B00NK332GS)
2. Apple iPhone 6 64 GB AT&T, Space Gray (B06XWNR8DB)
3. Apple iPhone 4S 16 GB AT&T, White (B005SSB0YO)
4. Apple iPhone 6s 16 GB US Domestic Warranty Unlocked Cellphone - Retail Packaging (Space Gray) (B015E8UBXS)
5. Apple iPhone 4 8 GB AT&T, White (B0074R1D0S)

QUERY: wireless earbuds with good battery life
1. Wireless Earbuds Bluetooth Earbuds 35H Cycle Playtime with Charging Case Ear Buds Wireless Stereo Earphones for iPhone/Android (B08H15SQ3H)
2. FOCUSPOWER F10 Mini Bluetooth Earbud Smallest Wireless Invisible Headphone with 6 Hour Playtime Car Headset with Mic for iPhone and Android Smart Phones(One Pcs) (B01M2ZOLLP)
3. Wireless Earbuds, Bluetooth 5.2 Headphones with Wireless Charging Case 1200mAh-60Hrs Play Time-Cell Phones Charging Function, Built-in Microphone IPX5 Waterproof Earphone for iOS/Android(Bright Black) (B0C778Z3RJ)
4. Micool Bluetooth Earpiece,30 Hours Talking Time,Noise Cancelling,12g Lightweight, Ha

## 4. Hybrid RRF Results

The hybrid retriever combines BM25 and FAISS rankings using Reciprocal Rank Fusion (RRF).

In [5]:
hybrid_results = {}

for query in queries:
    hybrid_results[query] = retriever.search(query, top_k=5)

for query in queries:
    print(f"\nQUERY: {query}")
    for rank, result in enumerate(hybrid_results[query], start=1):
        print(
            f"{rank}. {result['title']} "
            f"({result['parent_asin']})"
        )


QUERY: iPhone
1. Apple Compatible iPhone 5c Case iPhone 5c Protective Case Clear iPhone 5c Case iPhone 5c Cover Case iPhone 5c Green Case by iPhone 5c Silicone Case iPhone 5c Wallet Thin Case by Cable And Case (B00FJIZK8A)
2. Apple iPhone 6 64 GB AT&T, Silver (B00NK332GS)
3. iPhone 6 Plus Case, iPhone 6 Plus or 6S Plus Armor Cases 6 Plus Tough Rugged Shockproof Armorbox Dual Layer Hybrid Hard or Soft Slim Protective Case by Cable and Case by White Armor Case (B00NOGWXFC)
4. Apple iPhone 6 64 GB AT&T, Space Gray (B06XWNR8DB)
5. iPhone 6S Case, iPhone 6 Wallet Case, Firefish [Card Slots] [Kickstand] Flip Folio Wallet Case Synthetic Leather Shell Scratch Resistant Protective Cover for Apple iPhone 6/6S 4.7"-Butterfly (B01LW7QCKO)

QUERY: wireless earbuds with good battery life
1. Wireless Earbuds Bluetooth Earbuds 35H Cycle Playtime with Charging Case Ear Buds Wireless Stereo Earphones for iPhone/Android (B08H15SQ3H)
2. Wireless Earbuds, Bluetooth 5.2 Headphones with Wireless Charging Ca

## 5. Small Manual Relevance Assessment

The top five results from each retrieval method are manually assessed as relevant, partially relevant, or not relevant to the query.

### Relevance Labels

- **Relevant:** directly matches the query.
- **Partially relevant:** related, but does not fully satisfy the query.
- **Not relevant:** does not meaningfully match the query.

In [6]:
# Create assessment table for manual relevance labeling

assessment_rows = []

for query in queries:
    # BM25
    for rank, result in enumerate(bm25_results[query], start=1):
        product = retriever.product_lookup[result["parent_asin"]]
        assessment_rows.append({
            "query": query,
            "method": "BM25",
            "rank": rank,
            "parent_asin": result["parent_asin"],
            "title": product["title"]
        })

    # FAISS
    for rank, result in enumerate(faiss_results[query], start=1):
        product = retriever.product_lookup[result["parent_asin"]]
        assessment_rows.append({
            "query": query,
            "method": "FAISS",
            "rank": rank,
            "parent_asin": result["parent_asin"],
            "title": product["title"]
        })

    # Hybrid RRF
    for rank, result in enumerate(hybrid_results[query], start=1):
        assessment_rows.append({
            "query": query,
            "method": "Hybrid RRF",
            "rank": rank,
            "parent_asin": result["parent_asin"],
            "title": result["title"]
        })

assessment = pd.DataFrame(assessment_rows)

assessment

,query,method,rank,parent_asin,title
0,iPhone,BM25,1,B00FJIZK8A,Apple Compatible iPhone 5c Case iPhone 5c Protective Case Clear iPhone 5c Case iPhone 5c Cover C...
1,iPhone,BM25,2,B00NOGWXFC,"iPhone 6 Plus Case, iPhone 6 Plus or 6S Plus Armor Cases 6 Plus Tough Rugged Shockproof Armorbox..."
2,iPhone,BM25,3,B01LW7QCKO,"iPhone 6S Case, iPhone 6 Wallet Case, Firefish [Card Slots] [Kickstand] Flip Folio Wallet Case S..."
3,iPhone,BM25,4,B00NOGWZKK,"iPhone 6S Plus Case, iPhone 6 Plus Case by Cable And Case - Raised Screen Protector - Compatible..."
4,iPhone,BM25,5,B0C61PPHRM,"Small Portable Charger 5200mAh for iPhone, Ultra Compact 20W PD Fast Charging Power Bank,Cute Mi..."
...,...,...,...,...,...
85,earbuds with good battery life but poor Bluetooth connectivity is unacceptable,Hybrid RRF,1,B0C778Z3RJ,"Wireless Earbuds, Bluetooth 5.2 Headphones with Wireless Charging Case 1200mAh-60Hrs Play Time-C..."
86,earbuds with good battery life but poor Bluetooth connectivity is unacceptable,Hybrid RRF,2,B08H15SQ3H,Wireless Earbuds Bluetooth Earbuds 35H Cycle Playtime with Charging Case Ear Buds Wireless Stere...
87,earbuds with good battery life but poor Bluetooth connectivity is unacceptable,Hybrid RRF,3,B07CM5XVRL,"True Wireless Bluetooth Earbuds with Superior Sound, Easy-Pairing, Black Earphones in-Ear with C..."
88,earbuds with good battery life but poor Bluetooth connectivity is unacceptable,Hybrid RRF,4,B07M919FMW,"YUWISS Bluetooth Headset [36Hrs Playtime, 2 Batteries, V4.2] Wireless Bluetooth Earpiece for Cel..."


In [7]:
# Manually assigned relevance labels
relevance_labels = {
    # iPhone
    ("iPhone", "BM25", 1): "Not relevant",
    ("iPhone", "BM25", 2): "Not relevant",
    ("iPhone", "BM25", 3): "Not relevant",
    ("iPhone", "BM25", 4): "Not relevant",
    ("iPhone", "BM25", 5): "Not relevant",

    ("iPhone", "FAISS", 1): "Relevant",
    ("iPhone", "FAISS", 2): "Relevant",
    ("iPhone", "FAISS", 3): "Relevant",
    ("iPhone", "FAISS", 4): "Relevant",
    ("iPhone", "FAISS", 5): "Relevant",

    ("iPhone", "Hybrid RRF", 1): "Not relevant",
    ("iPhone", "Hybrid RRF", 2): "Relevant",
    ("iPhone", "Hybrid RRF", 3): "Not relevant",
    ("iPhone", "Hybrid RRF", 4): "Relevant",
    ("iPhone", "Hybrid RRF", 5): "Not relevant",

    # wireless earbuds with good battery life
    ("wireless earbuds with good battery life", "BM25", 1): "Relevant",
    ("wireless earbuds with good battery life", "BM25", 2): "Relevant",
    ("wireless earbuds with good battery life", "BM25", 3): "Partially relevant",
    ("wireless earbuds with good battery life", "BM25", 4): "Partially relevant",
    ("wireless earbuds with good battery life", "BM25", 5): "Partially relevant",

    ("wireless earbuds with good battery life", "FAISS", 1): "Relevant",
    ("wireless earbuds with good battery life", "FAISS", 2): "Partially relevant",
    ("wireless earbuds with good battery life", "FAISS", 3): "Relevant",
    ("wireless earbuds with good battery life", "FAISS", 4): "Partially relevant",
    ("wireless earbuds with good battery life", "FAISS", 5): "Partially relevant",

    ("wireless earbuds with good battery life", "Hybrid RRF", 1): "Relevant",
    ("wireless earbuds with good battery life", "Hybrid RRF", 2): "Relevant",
    ("wireless earbuds with good battery life", "Hybrid RRF", 3): "Relevant",
    ("wireless earbuds with good battery life", "Hybrid RRF", 4): "Partially relevant",
    ("wireless earbuds with good battery life", "Hybrid RRF", 5): "Relevant",

    # comfortable earbuds for running with good battery life
    ("comfortable earbuds for running with good battery life", "BM25", 1): "Relevant",
    ("comfortable earbuds for running with good battery life", "BM25", 2): "Relevant",
    ("comfortable earbuds for running with good battery life", "BM25", 3): "Relevant",
    ("comfortable earbuds for running with good battery life", "BM25", 4): "Partially relevant",
    ("comfortable earbuds for running with good battery life", "BM25", 5): "Partially relevant",

    ("comfortable earbuds for running with good battery life", "FAISS", 1): "Relevant",
    ("comfortable earbuds for running with good battery life", "FAISS", 2): "Partially relevant",
    ("comfortable earbuds for running with good battery life", "FAISS", 3): "Not relevant",
    ("comfortable earbuds for running with good battery life", "FAISS", 4): "Relevant",
    ("comfortable earbuds for running with good battery life", "FAISS", 5): "Partially relevant",

    ("comfortable earbuds for running with good battery life", "Hybrid RRF", 1): "Relevant",
    ("comfortable earbuds for running with good battery life", "Hybrid RRF", 2): "Relevant",
    ("comfortable earbuds for running with good battery life", "Hybrid RRF", 3): "Partially relevant",
    ("comfortable earbuds for running with good battery life", "Hybrid RRF", 4): "Not relevant",
    ("comfortable earbuds for running with good battery life", "Hybrid RRF", 5): "Relevant",

    # waterproof wireless earbuds for sports
    ("waterproof wireless earbuds for sports", "BM25", 1): "Relevant",
    ("waterproof wireless earbuds for sports", "BM25", 2): "Relevant",
    ("waterproof wireless earbuds for sports", "BM25", 3): "Partially relevant",
    ("waterproof wireless earbuds for sports", "BM25", 4): "Relevant",
    ("waterproof wireless earbuds for sports", "BM25", 5): "Not relevant",

    ("waterproof wireless earbuds for sports", "FAISS", 1): "Partially relevant",
    ("waterproof wireless earbuds for sports", "FAISS", 2): "Relevant",
    ("waterproof wireless earbuds for sports", "FAISS", 3): "Relevant",
    ("waterproof wireless earbuds for sports", "FAISS", 4): "Relevant",
    ("waterproof wireless earbuds for sports", "FAISS", 5): "Not relevant",

    ("waterproof wireless earbuds for sports", "Hybrid RRF", 1): "Relevant",
    ("waterproof wireless earbuds for sports", "Hybrid RRF", 2): "Relevant",
    ("waterproof wireless earbuds for sports", "Hybrid RRF", 3): "Relevant",
    ("waterproof wireless earbuds for sports", "Hybrid RRF", 4): "Partially relevant",
    ("waterproof wireless earbuds for sports", "Hybrid RRF", 5): "Not relevant",

    # iPhone case with good grip
    ("iPhone case with good grip", "BM25", 1): "Relevant",
    ("iPhone case with good grip", "BM25", 2): "Relevant",
    ("iPhone case with good grip", "BM25", 3): "Partially relevant",
    ("iPhone case with good grip", "BM25", 4): "Not relevant",
    ("iPhone case with good grip", "BM25", 5): "Partially relevant",

    ("iPhone case with good grip", "FAISS", 1): "Relevant",
    ("iPhone case with good grip", "FAISS", 2): "Relevant",
    ("iPhone case with good grip", "FAISS", 3): "Relevant",
    ("iPhone case with good grip", "FAISS", 4): "Relevant",
    ("iPhone case with good grip", "FAISS", 5): "Relevant",

    ("iPhone case with good grip", "Hybrid RRF", 1): "Partially relevant",
    ("iPhone case with good grip", "Hybrid RRF", 2): "Relevant",
    ("iPhone case with good grip", "Hybrid RRF", 3): "Relevant",
    ("iPhone case with good grip", "Hybrid RRF", 4): "Relevant",
    ("iPhone case with good grip", "Hybrid RRF", 5): "Relevant",

    # earbuds with battery + Bluetooth constraint
    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "BM25", 1): "Partially relevant",
    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "BM25", 2): "Partially relevant",
    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "BM25", 3): "Partially relevant",
    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "BM25", 4): "Partially relevant",
    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "BM25", 5): "Not relevant",

    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "FAISS", 1): "Partially relevant",
    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "FAISS", 2): "Partially relevant",
    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "FAISS", 3): "Partially relevant",
    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "FAISS", 4): "Partially relevant",
    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "FAISS", 5): "Partially relevant",

    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "Hybrid RRF", 1): "Partially relevant",
    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "Hybrid RRF", 2): "Partially relevant",
    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "Hybrid RRF", 3): "Partially relevant",
    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "Hybrid RRF", 4): "Partially relevant",
    ("earbuds with good battery life but poor Bluetooth connectivity is unacceptable", "Hybrid RRF", 5): "Partially relevant",
}

assessment["relevance"] = assessment.apply(
    lambda row: relevance_labels[
        (row["query"], row["method"], row["rank"])
    ],
    axis=1
)

assessment

,query,method,rank,parent_asin,title,relevance
0,iPhone,BM25,1,B00FJIZK8A,Apple Compatible iPhone 5c Case iPhone 5c Protective Case Clear iPhone 5c Case iPhone 5c Cover C...,Not relevant
1,iPhone,BM25,2,B00NOGWXFC,"iPhone 6 Plus Case, iPhone 6 Plus or 6S Plus Armor Cases 6 Plus Tough Rugged Shockproof Armorbox...",Not relevant
2,iPhone,BM25,3,B01LW7QCKO,"iPhone 6S Case, iPhone 6 Wallet Case, Firefish [Card Slots] [Kickstand] Flip Folio Wallet Case S...",Not relevant
3,iPhone,BM25,4,B00NOGWZKK,"iPhone 6S Plus Case, iPhone 6 Plus Case by Cable And Case - Raised Screen Protector - Compatible...",Not relevant
4,iPhone,BM25,5,B0C61PPHRM,"Small Portable Charger 5200mAh for iPhone, Ultra Compact 20W PD Fast Charging Power Bank,Cute Mi...",Not relevant
...,...,...,...,...,...,...
85,earbuds with good battery life but poor Bluetooth connectivity is unacceptable,Hybrid RRF,1,B0C778Z3RJ,"Wireless Earbuds, Bluetooth 5.2 Headphones with Wireless Charging Case 1200mAh-60Hrs Play Time-C...",Partially relevant
86,earbuds with good battery life but poor Bluetooth connectivity is unacceptable,Hybrid RRF,2,B08H15SQ3H,Wireless Earbuds Bluetooth Earbuds 35H Cycle Playtime with Charging Case Ear Buds Wireless Stere...,Partially relevant
87,earbuds with good battery life but poor Bluetooth connectivity is unacceptable,Hybrid RRF,3,B07CM5XVRL,"True Wireless Bluetooth Earbuds with Superior Sound, Easy-Pairing, Black Earphones in-Ear with C...",Partially relevant
88,earbuds with good battery life but poor Bluetooth connectivity is unacceptable,Hybrid RRF,4,B07M919FMW,"YUWISS Bluetooth Headset [36Hrs Playtime, 2 Batteries, V4.2] Wireless Bluetooth Earpiece for Cel...",Partially relevant


In [8]:
# Relevance statistics by retrieval method

relevance_stats = (
    assessment
    .groupby(["method", "relevance"])
    .size()
    .unstack(fill_value=0)
)

# Ensure consistent column order
relevance_stats = relevance_stats[
    ["Relevant", "Partially relevant", "Not relevant"]
]

# Add percentages
relevance_percent = (
    relevance_stats
    .div(relevance_stats.sum(axis=1), axis=0)
    .mul(100)
    .round(1)
)

print("Relevance Counts:")
display(relevance_stats)

print("\nRelevance Percentages:")
display(relevance_percent)

Relevance Counts:


relevance,Relevant,Partially relevant,Not relevant
method,,,
BM25,10,12,8
FAISS,17,11,2
Hybrid RRF,16,9,5



Relevance Percentages:


relevance,Relevant,Partially relevant,Not relevant
method,,,
BM25,33.3,40.0,26.7
FAISS,56.7,36.7,6.7
Hybrid RRF,53.3,30.0,16.7


## 7. Summary

The small manual assessment shows that BM25 is effective for keyword-specific queries, while FAISS provides stronger semantic matches. Hybrid RRF combines lexical and semantic retrieval and improves over BM25 in directly relevant results, but it can still return irrelevant products when product categories overlap or when queries contain complex constraints. In this evaluation sample, FAISS achieved the highest proportion of relevant results, while Hybrid RRF provided a balanced combination of lexical and semantic retrieval.